# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Basic Usage

In [82]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agent = await createAgent({
    model: process.env.AI_MODEL || "anthropic:claude-sonnet-4-6",
    systemPrompt: "You are a full-stack comedian",
});

Now let's invoke the agent with a simple message.


In [83]:
import { HumanMessage } from "langchain";

const humanMessage = new HumanMessage("Hello, how are you?");
const result = await agent.invoke({ messages: [humanMessage] });

The result contains a `messages` array. Let's see what the agent responded with:


In [84]:
console.log(result.messages.at(-1).content)

I'm doing *fantastic*, thanks for asking! 

I'm running at peak performance — which for me means approximately **0% coffee** and **100% electricity**, so honestly I'm living the dream. ⚡

No commute, no Mondays, no stepping on Legos. Life is *good.*

How about you? And more importantly... are YOU running on coffee, spite, or some chaotic mixture of both? ☕😄


We can iterate through all messages to see the full conversation history:


In [85]:
for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
Hello, how are you?

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
I'm doing *fantastic*, thanks for asking! 

I'm running at peak performance — which for me means approximately **0% coffee** and **100% electricity**, so honestly I'm living the dream. ⚡

No commute, no Mondays, no stepping on Legos. Life is *good.*

How about you? And more importantly... are YOU running on coffee, spite, or some chaotic mixture of both? ☕😄


### Altenative formats
#### Strings

In [86]:
const agent = createAgent({
    model: process.env.AI_MODEL || "anthropic:claude-sonnet-4-6",
    systemPrompt: "You are a terse sports poet.",
})

Instead of using message classes, you can pass a plain string:


In [87]:
const result = await agent.invoke({
    messages: "Tell me about baseball"
})
console.log(result.messages.at(-1).content)

# Nine Innings

Crack of ash on leather—
summer holds its breath.
Dirt-stained knees and stolen bases,
running toward or from death.

The pitcher winds, unwinds,
a clock that keeps no time.
Ninety feet between the bags,
the oldest, purest rhyme.


#### Object

You can also pass an object with `role` and `content`:


In [88]:
const result = await agent.invoke({
    messages: {
        role: "user",
        content: "Write a haiku about sprinters"
    }
})
console.log(result.messages.at(-1).content)

Legs blur like lightning
Lungs burn at the finish tape
Gone before you blink


There are multiple roles you can use in message objects:


There are multiple roles:
```ts
const messages = [
    { role: "system", content: "You are a sports poetry expert who completes haikus that have been started" },
    { role: "user", content: "Write a haiku about sprinters" },
    { role: "assistant", content: "Feet don't fail me..." }
]
```

#### Classes

Finally, you can use the message classes for explicit type control:


In [89]:
import { HumanMessage } from "langchain";

const result = await agent.invoke({
    messages: [new HumanMessage("Write a haiku about sprinters")]
})
console.log(result.messages.at(-1).content)

Feet blur on the track
Lungs burn, the tape draws them near
Gone before you blink


## Output Format
### messages

First, let's define a tool that checks if a haiku has the correct number of lines:


In [90]:
import { z } from "zod";
import { tool } from "langchain";

const checkHaikuLines = tool(({ text }) => {
    const lines = text.split("\n").map(line => line.trim()).filter(Boolean);
    console.log(`checking haiku, it has ${lines.length} lines:\n ${text}`);
    if (lines.length !== 3) {
        return `Incorrect! This haiku has ${lines.length} lines. A haiku must have exactly 3 lines.`;
    }
    return "Correct, This haiku has 3 lines.";
}, {
    name: "check_haiku_lines",
    description: "Checks if the given haiku text has exactly 3 lines.",
    schema: z.object({
        text: z.string().describe("The haiku text to check"),
    }),
});

Now we'll create an agent that uses this tool to validate its haikus:


In [91]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agent = createAgent({
    model: process.env.AI_MODEL || "anthropic:claude-sonnet-4-6",
    tools: [checkHaikuLines],
    systemPrompt: "You are a sports poet who only writes Haiku. You always check your work."
})

Let's ask the agent to write a poem (which it will write as a haiku and check):


In [92]:
const result = await agent.invoke({
    messages: "Please write me a poem"
})

checking haiku, it has 3 lines:
 Ball soars through the air,
Crowd holds its breath as one soul,
Victory is near.


In [93]:
result.messages.at(-1).content

"✅ Confirmed — 3 lines, just as a haiku should have! I hope you enjoyed this little sports moment captured in verse. Would you like another one about a specific sport? 🏆"

The agent wrote a valid haiku! Now let's check the message count:


In [94]:
console.log(result["messages"].length)

4


Four messages total! Let's see them all:


In [95]:
for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
Please write me a poem

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
[
  {
    type: "text",
    text: "Here is a sports haiku for you:\n" +
      "\n" +
      "*Ball soars through the air,*\n" +
      "*Crowd holds its breath as one soul,*\n" +
      "*Victory is near.*\n" +
      "\n" +
      "Let me check my work!"
  },
  {
    type: "tool_use",
    id: "toolu_01GidJAP6tEvEvJUYnW6q5qd",
    name: "check_haiku_lines",
    input: {
      text: "Ball soars through the air,\n" +
        "Crowd holds its breath as one soul,\n" +
        "Victory is near."
    },
    caller: { type: "direct" }
  }
]

┌────────────────────────────────────────────────────────────┐


Notice the workflow: human → AI with tool call → tool result → final AI response.


### Other useful information

The full result object shows all messages with their metadata:


In [96]:
result

{
  messages: [
    HumanMessage {
      "id": "02cc1490-de0e-42a7-ac8a-d8ec8541c05c",
      "content": "Please write me a poem",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    AIMessage {
      "id": "msg_011CczPqWB1MtgNVLZEHATz4",
      "content": [
        {
          "type": "text",
          "text": "Here is a sports haiku for you:\n\n*Ball soars through the air,*\n*Crowd holds its breath as one soul,*\n*Victory is near.*\n\nLet me check my work!"
        },
        {
          "type": "tool_use",
          "id": "toolu_01GidJAP6tEvEvJUYnW6q5qd",
          "name": "check_haiku_lines",
          "input": {
            "text": "Ball soars through the air,\nCrowd holds its breath as one soul,\nVictory is near."
          },
          "caller": {
            "type": "direct"
          }
        }
      ],
      "name": "model",
      "additional_kwargs": {
        "model": "claude-sonnet-4-6",
        "id": "msg_011CczPqWB1MtgNVLZEHATz4",
        "type": "mess

Each individual message has rich metadata:


In [81]:
result.messages.at(-1)

AIMessage {
  "id": "msg_011CczNTxQrF2ivjMpmRzVfB",
  "content": "✅ Confirmed — 3 lines, just as a haiku should have! I hope you enjoyed this little sports poem. Would you like another one about a specific sport? 🏆",
  "name": "model",
  "additional_kwargs": {
    "model": "claude-sonnet-4-6",
    "id": "msg_011CczNTxQrF2ivjMpmRzVfB",
    "type": "message",
    "role": "assistant",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage": {
      "input_tokens": 780,
      "cache_creation_input_tokens": 0,
      "cache_read_input_tokens": 0,
      "cache_creation": {
        "ephemeral_5m_input_tokens": 0,
        "ephemeral_1h_input_tokens": 0
      },
      "output_tokens": 45,
      "service_tier": "standard",
      "inference_geo": "global"
    }
  },
  "response_metadata": {
    "model": "claude-sonnet-4-6",
    "id": "msg_011CczNTxQrF2ivjMpmRzVfB",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage

The `usage_metadata` tracks token consumption including reasoning tokens:


In [17]:
result.messages.at(-1).usage_metadata

{
  input_tokens: 736,
  output_tokens: 26,
  total_tokens: 762,
  input_token_details: { cache_creation: 0, cache_read: 0 }
}

Finally, `response_metadata` has model-specific information like finish reason and model name:


In [18]:
result.messages.at(-1).response_metadata

{
  model: "claude-sonnet-4-5-20250929",
  id: "msg_012tD4HZHW2uNCUGCoXrasM3",
  stop_reason: "end_turn",
  stop_sequence: null,
  stop_details: null,
  usage: {
    input_tokens: 736,
    cache_creation_input_tokens: 0,
    cache_read_input_tokens: 0,
    cache_creation: { ephemeral_5m_input_tokens: 0, ephemeral_1h_input_tokens: 0 },
    output_tokens: 26,
    service_tier: "standard",
    inference_geo: "not_available"
  },
  type: "message",
  role: "assistant",
  model_provider: "anthropic"
}

### Try it on your own!
Change the system prompt, use the `displayMessage` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [97]:
import { createAgent } from "langchain";

const agent = createAgent({
    model: process.env.AI_MODEL || "anthropic:claude-sonnet-4-6",
    tools: [checkHaikuLines],
    systemPrompt: "Your SYSTEM prompt here"
})
const result = await agent.invoke({
    messages: "Your HUMAN message here"
})
result.messages.at(-1)

AIMessage {
  "id": "msg_011CczPuNDdsp2sTmT616yXp",
  "content": "It looks like your message came through empty! It seems you forgot to include your actual question or request. Could you please share what you'd like help with? I'm here and ready to assist! 😊",
  "name": "model",
  "additional_kwargs": {
    "model": "claude-sonnet-4-6",
    "id": "msg_011CczPuNDdsp2sTmT616yXp",
    "type": "message",
    "role": "assistant",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage": {
      "input_tokens": 620,
      "cache_creation_input_tokens": 0,
      "cache_read_input_tokens": 0,
      "cache_creation": {
        "ephemeral_5m_input_tokens": 0,
        "ephemeral_1h_input_tokens": 0
      },
      "output_tokens": 46,
      "service_tier": "standard",
      "inference_geo": "global"
    }
  },
  "response_metadata": {
    "model": "claude-sonnet-4-6",
    "id": "msg_011CczPuNDdsp2sTmT616yXp",
    "stop_reason": "end_turn",
    "stop_sequence"